# PageIndex — Vectorless RAG Demo
This notebook demonstrates how to:
1. Submit a PDF and build a structural tree index (once only)
2. Query the indexed document using LLM-guided tree navigation
3. Generate and print answers grounded in exact document sections

## Imports & API Key Setup

In [2]:
import os
import time
import json
from pageindex import PageIndexClient
import openai

from google import genai

# ── Configure your API keys here ──────────────────────────────────────────────
gemini = genai.Client(api_key="xxxxxxx")

PAGEINDEX_API_KEY = "xxxxxxxx"

# Initialize clients
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

print("✅ Clients initialised successfully.")

✅ Clients initialised successfully.


## Define the PDF Path
Set `PDF_PATH` to the local path of your PDF. The document is indexed **only once** — subsequent cells reuse the stored `doc_id`.

In [3]:
# Path to your PDF document
PDF_PATH = "2412.19437v2.pdf"

# This variable persists across cells so we never re-index the same document
doc_id = None

print(f"📄 Target document : {PDF_PATH}")

📄 Target document : 2412.19437v2.pdf


## Submit Document & Build the Structural Tree Index
> ⚠️ **Run this cell only once.** It submits the PDF to PageIndex and waits for the structural tree to be built. The `doc_id` is saved so you can skip this cell on subsequent runs.

In [4]:
# Guard: skip indexing if we already have a doc_id
if doc_id is not None:
    print(f"⏭️  Document already indexed — skipping.  doc_id = {doc_id}")
else:
    print("📤 Step 1 — Submitting document to PageIndex...")
    submit_resp = pi_client.submit_document(PDF_PATH)

    # ── Inspect the raw submission response ───────────────────────────────────
    print("\n📦 Raw submission response:")
    print(json.dumps(submit_resp, indent=2))

    doc_id = submit_resp.get("doc_id")
    status = submit_resp.get("status", "unknown")

    print(f"\n🆔 Document ID   : {doc_id}")
    print(f"🔄 Initial status : {status}")

    # ── Wait for the tree to be built ─────────────────────────────────────────
    # PageIndexClient does not expose a status-polling method.
    # We wait a fixed period — increase WAIT_SECONDS for larger documents.
    WAIT_SECONDS = 30
    print(f"\n⏳ Step 2 — Waiting {WAIT_SECONDS}s for the structural tree to be built...")

    for attempt in range(1, 60):
        tree_result = pi_client.get_tree(doc_id)
        status = tree_result.get("status", "unknown")
        print(f"  [{attempt:02d}] status={status}")
        if status == "completed":
            break
        time.sleep(5)

    print(f"\n✅ Indexing complete!  doc_id = {doc_id}")

📤 Step 1 — Submitting document to PageIndex...

📦 Raw submission response:
{
  "doc_id": "pi-cmmnd994403z10qqnup5q8xd5"
}

🆔 Document ID   : pi-cmmnd994403z10qqnup5q8xd5
🔄 Initial status : unknown

⏳ Step 2 — Waiting 30s for the structural tree to be built...
   30s remaining...
   25s remaining...
   20s remaining...
   15s remaining...
   10s remaining...
   5s remaining...

✅ Indexing complete!  doc_id = pi-cmmnd994403z10qqnup5q8xd5


## Explore the Index Structure
Before querying, inspect the top-level sections PageIndex identified in the document.

In [20]:
# submit_resp is already in memory from Cell 4
print(f"🔍 Submission details for doc_id: {doc_id}\n")
print("-" * 60)
for key, value in submit_resp.items():
    print(f"  {key:25s} : {value}")
print("-" * 60)
print("\n✅ The document tree has been built internally by PageIndex.")
print("   Queries in the next cells will navigate this tree directly.")

🔍 Submission details for doc_id: pi-cmmnd994403z10qqnup5q8xd5

------------------------------------------------------------
  doc_id                    : pi-cmmnd994403z10qqnup5q8xd5
------------------------------------------------------------

✅ The document tree has been built internally by PageIndex.
   Queries in the next cells will navigate this tree directly.


## Define the Query Helper Function

In [22]:
# Cell 6 — Query using PageIndex's built-in Chat API
def run_rag(query: str) -> str:
    response = pi_client.chat_completions(
        messages=[{"role": "user", "content": query}],
        doc_id=doc_id
    )
    answer = response["choices"][0]["message"]["content"]
    print("=" * 60)
    print(f"QUERY  : {query}")
    print("=" * 60)
    print(answer)
    return answer

## Run a Query

In [33]:
# question = "What is the total parameter count of DeepSeek-V3-Base?"
question = "What is the architecture of DeepSeek-V3?"

# answer = run_vectorless_rag(question)
answer = run_rag(question)

QUERY  : What is the architecture of DeepSeek-V3?
I'll help you find the architecture details of DeepSeek-V3. Let me first check the document structure to locate the relevant sections.Now let me extract the architecture section details:## DeepSeek-V3 Architecture

DeepSeek-V3 is a **671B parameter Mixture-of-Experts (MoE)** language model built on the Transformer framework with three key architectural innovations:

### 1. **Multi-Head Latent Attention (MLA)**
MLA enables efficient inference through low-rank compression:
- **KV compression**: Compresses attention keys and values into compact latent vectors, significantly reducing KV cache during generation
- **Query compression**: Also compresses queries to reduce activation memory during training
- Only compressed latent vectors (c^KV and k^R) need to be cached, dramatically improving memory efficiency while maintaining performance comparable to standard multi-head attention

### 2. **DeepSeekMoE**
An economical MoE architecture featur

In [34]:
from IPython.display import Markdown, display
display(Markdown(answer))

I'll help you find the architecture details of DeepSeek-V3. Let me first check the document structure to locate the relevant sections.Now let me extract the architecture section details:## DeepSeek-V3 Architecture

DeepSeek-V3 is a **671B parameter Mixture-of-Experts (MoE)** language model built on the Transformer framework with three key architectural innovations:

### 1. **Multi-Head Latent Attention (MLA)**
MLA enables efficient inference through low-rank compression:
- **KV compression**: Compresses attention keys and values into compact latent vectors, significantly reducing KV cache during generation
- **Query compression**: Also compresses queries to reduce activation memory during training
- Only compressed latent vectors (c^KV and k^R) need to be cached, dramatically improving memory efficiency while maintaining performance comparable to standard multi-head attention

### 2. **DeepSeekMoE**
An economical MoE architecture featuring:
- **Expert structure**: Combines shared experts (always activated) and routed experts (selectively activated via top-K routing)
- **Auxiliary-loss-free load balancing**: A novel strategy using dynamic bias terms to balance expert loads without performance degradation
  - Bias terms are adjusted at each training step based on expert load
  - Overloaded experts get decreased bias, underloaded experts get increased bias
- **Complementary sequence-wise loss**: A minimal auxiliary loss to prevent extreme imbalance within individual sequences
- **Node-limited routing**: Restricts token routing to at most M nodes to reduce communication costs
- **No token dropping**: Effective load balancing eliminates the need to drop tokens during training or inference

### 3. **Multi-Token Prediction (MTP)**
A training objective that enhances performance:
- Predicts multiple future tokens at each position (not just the next one)
- Uses **sequential prediction** with complete causal chains (unlike parallel approaches)
- Employs D sequential modules, each with shared embedding/output layers, a Transformer block, and projection matrix
- Densifies training signals and enables pre-planning of representations
- Can be repurposed for speculative decoding during inference

The architecture builds upon DeepSeek-V2 while introducing the auxiliary-loss-free balancing strategy as a major improvement for better performance-efficiency tradeoff.